In [33]:
# Import the necessary libraries
import pandas as pd
import numpy as np
import re
import pickle
import time
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Load the datasets
rent = pd.read_csv("../Data/rent.csv")
sale = pd.read_csv("../Data/sale.csv")

In [3]:
# Display the shape of each dataset 
print("Rental Dataset Shape:", rent.shape) 
print("Sale Dataset Shape:", sale.shape)

Rental Dataset Shape: (9853, 4)
Sale Dataset Shape: (9784, 4)


In [4]:
# Display the first five rows 
print("\nRental Dataset Preview:") 
display(rent.head())
print("\nSales Dataset Preview:") 
display(sale.head())


Rental Dataset Preview:


,Price,Pid,Property_name,Address
0,"400,000 / year",H1190,1 bedroom mini flat Flat / Apartment for rent ...,Fidiso Estate Abijo Ajah Lagos
1,"14,000,000 / year",G7398,3 bedroom Flat / Apartment for rent Gerrard Ge...,Gerrard Gerrard Gerard Road Ikoyi Lagos
2,"200,000 / year",H5843,1 bedroom mini flat Flat / Apartment for rent ...,Obadia Akoka Yaba Lagos
3,"800,000 / year",H1194,2 bedroom Flat / Apartment for rent Iju Ishaga...,surulere Iju Ishaga Road Gbaja Surulere Lagos
4,"300,000 / year",H5842,1 bedroom mini flat Flat / Apartment for rent ...,Close to Unilag Abule Oja Yaba Lagos



Sales Dataset Preview:


,Price,Pid,Property_name,Address
0,"2,500,000",E9857,Land for sale Eleranigbe Eleranigbe Ibeju-Lekk...,Eleranigbe Eleranigbe Eleranigbe Ibeju Lekki L...
1,"30,000,000","17,H2630",667.0 square meter Land for sale opebi street ...,Opebi street opebi street Opebi Ikeja Lagos
2,"3,500,000,000","10,18,H4485",3 bedroom House for sale Victoria Island Exten...,Victoria Island Lagos Victoria Island Extensio...
3,"340,000,000",H4413,5 bedroom Duplex for sale Lekki Phase 1 Lekki ...,off afrika lane Lekki Phase 1 Lekki Lagos
4,"28,000,000",G8554,3 bedroom Flat / Apartment for sale Igando Iko...,NEAR IGANDO B/STOP Igando Ikotun Igando Lagos


In [5]:
# Inspect data types and non-null counts 
print("\nRental Dataset Information:") 
rent.info()
print("\nSales Dataset Information:") 
sale.info()


Rental Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 9853 entries, 0 to 9852
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Price          9853 non-null   str  
 1   Pid            9853 non-null   str  
 2   Property_name  9853 non-null   str  
 3   Address        9853 non-null   str  
dtypes: str(4)
memory usage: 1.4 MB

Sales Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 9784 entries, 0 to 9783
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Price          9784 non-null   str  
 1   Pid            9784 non-null   str  
 2   Property_name  9784 non-null   str  
 3   Address        9784 non-null   str  
dtypes: str(4)
memory usage: 1.4 MB


In [6]:
# Checking for missing values 
print("\nMissing Values in Rental Dataset:") 
print(rent.isnull().sum()) 
print("\nMissing Values in Sales Dataset:") 
print(sale.isnull().sum())


Missing Values in Rental Dataset:
Price            0
Pid              0
Property_name    0
Address          0
dtype: int64

Missing Values in Sales Dataset:
Price            0
Pid              0
Property_name    0
Address          0
dtype: int64


In [7]:
# Check for duplicate rows 
print("\nDuplicate Rows in Rental Dataset:", 
      rent.duplicated().sum()) 
print("Duplicate Rows in Sale Dataset:", 
      sale.duplicated().sum())


Duplicate Rows in Rental Dataset: 3
Duplicate Rows in Sale Dataset: 1


In [8]:
# Find duplicated property IDs in the rental dataset 
rent_duplicate_ids = rent.loc[ rent["Pid"].duplicated(keep=False), "Pid" ] 
print("Duplicated Property IDs in Rental Dataset:") 
print(rent_duplicate_ids)

sale_duplicate_ids = sale.loc[ sale["Pid"].duplicated(keep=False), "Pid" ] 
print("Duplicated Property IDs in Sale Dataset:") 
print(sale_duplicate_ids)

Duplicated Property IDs in Rental Dataset:
3273    G4171
3274    G4171
4404    G6707
4405    G6707
6956    G0258
6957    G0258
Name: Pid, dtype: str
Duplicated Property IDs in Sale Dataset:
604    H4190
605    H4190
Name: Pid, dtype: str


In [9]:
# Examine the number of unique values in each column 
print("\nUnique Values in Rental Dataset:") 
print(rent.nunique())
print("\nUnique Values in Sale Dataset:") 
print(sale.nunique())


Unique Values in Rental Dataset:
Price             388
Pid              9850
Property_name    4762
Address          5945
dtype: int64

Unique Values in Sale Dataset:
Price             581
Pid              9783
Property_name    5452
Address          7369
dtype: int64


In [14]:
# CLEAN PRICE
# Price like "400,000 / year" -> 400000
def clean_price(x):
    if pd.isna(x): return np.nan
    s = str(x).replace(',', '').replace('₦','')
    m = re.search(r'(\d+(?:\.\d+)?)', s)
    return float(m.group(1)) if m else np.nan
rent['price'] = rent['Price'].apply(clean_price)
print(rent['price'].describe())

count    9.853000e+03
mean     2.690928e+06
std      1.570867e+07
min      2.000000e+00
25%      5.000000e+05
50%      1.000000e+06
75%      2.500000e+06
max      1.100000e+09
Name: price, dtype: float64


In [16]:
# FEATURE ENGINEERING
# Bedrooms: "3 bedroom Flat..." -> 3
rent['bedrooms'] = rent['Property_name'].str.lower().str.extract(r'(\d+)\s*bedroom')[0].fillna(0).astype(int)


In [17]:
# Property Type
def get_type(s):
    s = str(s).lower()
    if 'self contain' in s: return 'Self Contain'
    if 'duplex' in s: return 'Duplex'
    if 'bungalow' in s: return 'Bungalow'
    if 'detached' in s: return 'House'
    if 'flat' in s or 'apartment' in s: return 'Flat'
    if 'house' in s: return 'House'
    if 'office' in s: return 'Office'
    if 'land' in s: return 'Land'
    if 'shop' in s: return 'Shop'
    return 'Other'
rent['prop_type'] = rent['Property_name'].apply(get_type)

In [49]:
# Area: First part of Address
# "Fidiso Estate Abijo Ajah Lagos" -> we take last major area
# Better: Search for known Lagos areas inside Address
SPECIFIC_AREAS = ['Banana Island','Victoria Island','Lekki Phase 1','Oniru','Ikoyi',
                  'Chevron','Ajah','Sangotedo','Ibeju','Abijo','Awoyaya',
                  'Yaba','Akoka','Surulere','Ikeja','Opebi','Magodo','Ogba',
                  'Ojodu','Ikorodu','Shomolu','Kosofe','Isolo','Okota','Ilupeju',
                  'Lekki','Apapa','Maryland','Gbagada','Alimosho','Epe']
def get_area_v2(row):
    text = str(row['Property_name']) + ' ' + str(row['Address'])
    text_low = text.lower()
    for a in SPECIFIC_AREAS:
        if a.lower() in text_low:
            return a
    return 'Other'
rent['area'] = rent.apply(get_area_v2, axis=1)
top = rent['area'].value_counts().head(50).index.tolist()
rent['area'] = rent['area'].apply(lambda x: x if x in top else 'Other')

In [50]:
print(rent['area'].value_counts())

area
Yaba               1135
Other               918
Ikeja               834
Lekki Phase 1       780
Lekki               667
Ajah                665
Surulere            632
Ikoyi               480
Gbagada             475
Victoria Island     462
Magodo              411
Ojodu               328
Ogba                292
Shomolu             289
Isolo               228
Okota               216
Chevron             190
Ikorodu             173
Maryland            166
Kosofe              148
Alimosho            119
Banana Island        83
Ilupeju              65
Apapa                32
Ibeju                30
Sangotedo            26
Oniru                 4
Akoka                 3
Awoyaya               2
Name: count, dtype: int64


In [51]:
# Hidden features
low = rent['Property_name'].str.lower()
rent['is_mini'] = low.str.contains('mini', na=False).astype(int)  # mini flat = cheaper
rent['is_new'] = low.str.contains('newly|new |serviced', na=False).astype(int) # newly built = expensive
rent['is_furnished'] = low.str.contains('furnished', na=False).astype(int)

In [74]:
# 1. See all areas - make we see wetin dey there
print(df['area'].value_counts().head(50))

print("\n--- Check Address ---")
print(df[df['Address'].astype(str).str.contains('command', case=False, na=False)]['Address'].value_counts().head(20))

print("\n--- Check Property_name ---")
print(df[df['Property_name'].astype(str).str.contains('command', case=False, na=False)]['Property_name'].value_counts().head(20))

print("\n--- Check full row for Command anywhere ---")
# Search all text columns
for col in ['Property_name', 'Address', 'area', 'prop_type']:
    mask = df[col].astype(str).str.contains('command', case=False, na=False)
    print(col, "->", mask.sum(), "rows with Command")

area
Yaba               1091
Other               874
Lekki Phase 1       749
Ikeja               741
Lekki               660
Ajah                654
Surulere            617
Gbagada             468
Ikoyi               459
Victoria Island     407
Magodo              402
Ojodu               325
Shomolu             285
Ogba                282
Isolo               219
Okota               215
Chevron             186
Ikorodu             163
Maryland            161
Kosofe              147
Alimosho            118
Banana Island        82
Ilupeju              57
Sangotedo            26
Ibeju                25
Apapa                21
Oniru                 4
Akoka                 3
Awoyaya               2
Name: count, dtype: int64

--- Check Address ---
Address
Command Ipaja Lagos Ipaja Ipaja Lagos                                           10
command Ipaja Road Ipaja Lagos                                                   3
Command Ipaja. Ipaja Road Ipaja Lagos                                       

In [75]:
# Checking where Command rows is
mask_command = df['Address'].astype(str).str.contains('command', case=False, na=False)
print("Before fix:")
print(df[mask_command]['area'].value_counts())
print("Total Command rows:", mask_command.sum())

# FIX: If Address contains 'command', force area to Command Ipaja
df.loc[mask_command, 'area'] = 'Command Ipaja'

print("\nAfter fix:")
print(df['area'].value_counts().head(30))
print("Now Command Ipaja count:", (df['area'] == 'Command Ipaja').sum())

Before fix:
area
Other       37
Alimosho     1
Name: count, dtype: int64
Total Command rows: 38

After fix:
area
Yaba               1091
Other               837
Lekki Phase 1       749
Ikeja               741
Lekki               660
Ajah                654
Surulere            617
Gbagada             468
Ikoyi               459
Victoria Island     407
Magodo              402
Ojodu               325
Shomolu             285
Ogba                282
Isolo               219
Okota               215
Chevron             186
Ikorodu             163
Maryland            161
Kosofe              147
Alimosho            117
Banana Island        82
Ilupeju              57
Command Ipaja        38
Sangotedo            26
Ibeju                25
Apapa                21
Oniru                 4
Akoka                 3
Awoyaya               2
Name: count, dtype: int64
Now Command Ipaja count: 38


In [76]:
# FILTER FOR V1
# Remove Land, Shop, Office - they are not house rent
# Remove outliers: <50k and >50M
df = rent[(rent['price'] >= 50000) & (rent['price'] <= 50000000)].copy()
df = df[~df['prop_type'].isin(['Land','Shop'])]
print(f"After filter: {df.shape} - dropped {len(rent)-len(df)} rows")


After filter: (9443, 11) - dropped 410 rows


In [83]:
# FIX - Must run BEFORE training
mask_command = df['Address'].astype(str).str.contains('command', case=False, na=False)
df.loc[mask_command, 'area'] = 'Command Ipaja'
print("Command Ipaja count:", (df['area'] == 'Command Ipaja').sum())

Command Ipaja count: 38


In [84]:
# MODEL - Predict log(price) because price is skewed
df['log_price'] = np.log1p(df['price'])
feature_cols = ['bedrooms','prop_type','area','is_mini','is_new','is_furnished']
X = pd.get_dummies(df[feature_cols], columns=['prop_type','area'])
y = df['log_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

model = xgb.XGBRegressor(n_estimators=800, max_depth=5, learning_rate=0.03,
                         subsample=0.8, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

pred = np.expm1(model.predict(X_test))
true = np.expm1(y_test)

mae = mean_absolute_error(true, pred)
rmse = np.sqrt(mean_squared_error(true, pred))
r2 = r2_score(true, pred)
mape = np.mean(np.abs(true-pred)/true)*100
within20 = np.mean(np.abs(true-pred)/true <= 0.20)*100
within30 = np.mean(np.abs(true-pred)/true <= 0.30)*100

print(f"\n=== RENT MODEL V1 RESULTS ===")
print(f"MAE: ₦{mae:,.0f}")
print(f"RMSE: ₦{rmse:,.0f}")
print(f"R2: {r2:.3f}")
print(f"MAPE: {mape:.1f}%")
print(f"Within 20%: {within20:.1f}%")
print(f"Within 30%: {within30:.1f}%")



=== RENT MODEL V1 RESULTS ===
MAE: ₦879,783
RMSE: ₦2,283,612
R2: 0.587
MAPE: 59.8%
Within 20%: 42.3%
Within 30%: 58.2%


In [85]:
# Baseline:
med = np.median(np.expm1(y_train))
print(f"Median baseline MAE: ₦{mean_absolute_error(true, [med]*len(true)):,.0f} - Our model must beat this")

Median baseline MAE: ₦1,710,549 - Our model must beat this


In [86]:
# SAVE
pickle.dump(model, open('rent_model.pkl','wb'))
pickle.dump(list(X.columns), open('rent_feature_cols.pkl','wb'))
df.to_csv('rent_clean.csv', index=False)
print("Saved: rent_model.pkl, rent_feature_cols.pkl, rent_clean.csv ✅")


Saved: rent_model.pkl, rent_feature_cols.pkl, rent_clean.csv ✅
